# День 08. Упражнение 03
# Переобучение

## 0. Импорты

In [127]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from collections import Counter

## 1. Предобработка

1. Прочитай файл `dayofweek.csv` в DataFrame.

In [104]:
df = pd.read_csv("../data/dayofweek.csv")
df.shape

(1686, 44)

In [105]:
y = df["dayofweek"]
X = df.drop(columns=["dayofweek"])

2. Используя `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, получи `X_train`, `y_train`, `X_test`, `y_test`.

In [106]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21
)
df_compare = pd.DataFrame(
    {
        "Train": y_train.value_counts(normalize=True),
        "Test": y_test.value_counts(normalize=True),
    }
)

3. С помощью, например, `value_counts()` проверь, похожее ли распределение классов в train и test.

4. Добавь параметр `stratify=` и проверь распределение снова — теперь оно должно быть более-менее одинаковым в обоих наборах данных.

In [107]:
# С параметром stratify=y: Алгоритм сначала считает пропорции дней недели во всем исходном датасете, а затем жестко следит за тем, чтобы и в Train (80% строк), и в Test (20% строк) эти пропорции сохранились в точности до процента.
# простыми словами: главный смысл стратификации — сделать Train и Test одинаковыми между собой
# почему это важно?  напр, у нас есть дисбаланс классов: чт — очень популярный день, а сб. и вскр. — редкие). Параметр страхует код от случайных перекосов при делении, что гарантирует, что в тестовую и трейн выборку попадет та же доля более редкого класса.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
df_compare["Train stratified"] = y_train.value_counts(normalize=True)
df_compare["Test stratified"] = y_test.value_counts(normalize=True)

print(
    "стратификация гарантирует, что в тестовую и трейн выборку попадет\nпрактически одинаковая доля из каждого класса"
)
df_compare

стратификация гарантирует, что в тестовую и трейн выборку попадет
практически одинаковая доля из каждого класса


,Train,Test,Train stratified,Test stratified
dayofweek,,,,
0,0.077893,0.091716,0.080861,0.079882
1,0.164688,0.153846,0.162463,0.162722
2,0.092730,0.071006,0.088279,0.088757
3,0.232196,0.245562,0.234421,0.236686
4,0.059347,0.071006,0.061573,0.062130
5,0.160237,0.162722,0.160979,0.159763
6,0.212908,0.204142,0.211424,0.210059


## 2. Базовые модели

1. Обучи те же самые базовые модели из предыдущего упражнения и вычисли accuracy, используя тестовый набор данных со стратификацией.
2. Все ли модели показали похожие значения метрики? Какая из них показала наибольшее отличие по сравнению с предыдущим упражнением? Напиши ответ в markdown-ячейке в конце раздела.

### a. Логистическая регрессия

In [129]:
solvers = ["newton-cg", "saga", "lbfgs"]

lr_accuracy = {}
trained_lr_models = {}

for s in solvers:
    model = LogisticRegression(
        solver=s, fit_intercept=False, random_state=21, max_iter=1000
    )
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    lr_accuracy[s] = acc
    trained_lr_models[s] = model
    # print(f"{f'Точность с {s}:':<22}{acc}")

most_common_acc = Counter(lr_accuracy.values()).most_common(1)[0][0]
print(f"Accuracy для модели логистической регресии: {most_common_acc}")

Accuracy для модели логистической регресии: 0.6272189349112426


In [137]:
# разбор most_common_acc = Counter(lr_accuracy.values()).most_common(1)[0][0]
lr_accuracy = {"newton-cg": 0.6272, "saga": 0.6272, "lbfgs": 0.6331}
print(lr_accuracy.values())
print(Counter(lr_accuracy.values()))
print(Counter(lr_accuracy.values()).most_common(1))  # кортеж внутри списка
print(Counter(lr_accuracy.values()).most_common(1)[0][0])

dict_values([0.6272, 0.6272, 0.6331])
Counter({0.6272: 2, 0.6331: 1})
[(0.6272, 2)]
0.6272


### b. SVM

In [138]:
# Запуск прямого SVC без использования мета-классификатора OneVsRestClassifier
model_svc_final = SVC(kernel="linear", random_state=21, probability=True)
model_svc_final.fit(X_train, y_train)

accuracy_svc_test = accuracy_score(y_test, model_svc_final.predict(X_test))
print(f"Accuracy для модели SVC: {accuracy_svc_test}")

Accuracy для модели SVC: 0.7159763313609467


### c. Дерево решений

In [139]:
# Accuracy для DT в ex02 была 0.5516014234875445
model_dt = DecisionTreeClassifier(max_depth=4, random_state=21)
model_dt.fit(X_train, y_train)
y_pred_dt_test = model_dt.predict(X_test)
accuracy_dt_test = accuracy_score(y_test, y_pred_dt_test)
print(f"Accuracy для модели DT: {accuracy_dt_test}")

Accuracy для модели DT: 0.5295857988165681


### d. Случайный лес

In [140]:
model_rf = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21)
model_rf.fit(X_train, y_train)
y_pred_rf_test = model_rf.predict(X_test)
accuracy_rf_test = accuracy_score(y_test, y_pred_rf_test)
accuracy_rf_test

0.9289940828402367

## 3. Кросс-валидация

Можно было бы “поиграть” с параметрами модели, чтобы добиться более высокой точности на тестовом наборе, но так делать не принято. Такая тактика снова приводит к переобучению. Тестовый набор предназначен только для финальной проверки качества модели.

Однако есть другой способ решения этой задачи — кросс-валидация. Он не использует тестовый набор, а создает дополнительное разбиение обучающего набора. Существует несколько способов реализации, но общая идея одна: появляется валидационный набор, который используется для подбора гиперпараметров.

1. Используя `cross_val_score` с параметром `cv=10`, вычисли среднее значение accuracy и стандартное отклонение для каждой модели, использованной ранее (логистическая регрессия с `solver='liblinear'`, SVC, дерево решений, случайный лес).

### a. Логистическая регрессия

### b. SVM

### c. Дерево решений

### d. Случайный лес

## 4. Оптимизация

1. Выбери лучшую модель и немного поиграй с её параметрами на кросс-валидации, чтобы найти достаточно хорошие параметры или их комбинацию.
2. Вычисли accuracy для финальной модели на тестовом наборе данных.
3. Построй график, отображающий 10 самых важных признаков для этой модели.
4. Сохрани модель с помощью `joblib`.
5. Загрузи модель, сделай предсказания для тестового набора и вычисли accuracy.